# Byte-Level N-gram Analysis

Analyze the `mistobaan/fineweb10B_bytes` dataset — raw UTF-8 bytes stored in the same shard format as sp1024.
We compute unigram, bigram, trigram, and 4-gram statistics and compare storage/coverage to the sp1024 tokenizer variant.

In [ ]:
import glob
import numpy as np
from pathlib import Path
from collections import Counter
import matplotlib.pyplot as plt
from efficient_byte_tokenizer import (
    EfficientByteTokenizer,
    ByteCategory,
    OtherTokenStrategy,
)

DATA_PATH = "data/datasets/fineweb10B_bytes"
train_pattern = f"{DATA_PATH}/fineweb_train_*.bin"
val_pattern = f"{DATA_PATH}/fineweb_val_*.bin"

# --- Tokenizer configuration ---
# Set to None for raw byte representation (V=256), or configure:
_eff_tok = EfficientByteTokenizer()  # default: all used bytes, V=208
_eff_tok = EfficientByteTokenizer(
    fold=ByteCategory.UPPERCASE
)  # default: case insensitive
# _eff_tok = EfficientByteTokenizer(keep=ByteCategory.LETTER)  # letters only, drop rest
# _eff_tok = EfficientByteTokenizer(
#     keep=ByteCategory.LETTER,
#     fold=ByteCategory.UPPERCASE,
#     other=OtherTokenStrategy.BOUNDARY,
# )
# _eff_tok = EfficientByteTokenizer(keep={ByteCategory.LETTER, ByteCategory.DIGIT})
# _eff_tok = None                                              # raw bytes, V=256

if _eff_tok is not None:
    print(_eff_tok.describe())
else:
    print("Using raw byte representation (vocab_size=256)")

## Load byte data from shards
Uses the same header format as sp1024 (magic 20240520, version 1, uint16 storage with values 0-255).

In [ ]:
MAX_TOKENS_TRAIN = 500_000_000  # tokens for building n-gram tables
MAX_TOKENS_VAL = 10_000_000  # held-out tokens for BPB evaluation
MAX_SHARDS = 11


def load_data_shard(file: Path) -> np.ndarray:
    """Load a shard, returning uint8 token array."""
    header_bytes = 256 * np.dtype("<i4").itemsize
    header = np.fromfile(file, dtype="<i4", count=256)
    assert header[0] == 20240520 and header[1] == 1, f"Bad header in {file}"
    num_tokens = int(header[2])
    tokens = np.fromfile(file, dtype="<u2", count=num_tokens, offset=header_bytes)
    assert tokens.max() <= 255, f"Values exceed byte range in {file}"
    return tokens.astype(np.uint8)


# --- Load training data ---
files = sorted(glob.glob(train_pattern))[:MAX_SHARDS]
print(f"Found {len(files)} train shards")
for f in files:
    print(f"  {f}  ({Path(f).stat().st_size / 1e6:.1f} MB)")

train_tokens = np.concatenate([load_data_shard(Path(f)) for f in files])
if MAX_TOKENS_TRAIN > 0 and len(train_tokens) > MAX_TOKENS_TRAIN:
    train_tokens = train_tokens[:MAX_TOKENS_TRAIN]
print(f"\nTrain bytes loaded: {len(train_tokens):,}")

# --- Load validation data from dedicated val shard ---
val_files = sorted(glob.glob(val_pattern))
print(f"\nFound {len(val_files)} val shards")
val_tokens = np.concatenate([load_data_shard(Path(f)) for f in val_files])
if MAX_TOKENS_VAL > 0 and len(val_tokens) > MAX_TOKENS_VAL:
    val_tokens = val_tokens[:MAX_TOKENS_VAL]
print(f"Val bytes loaded:   {len(val_tokens):,}")

# --- Remap and filter if using efficient byte tokenizer ---
if _eff_tok is not None:
    train_tokens = _eff_tok.filter_stream(_eff_tok.remap_byte_array(train_tokens))
    val_tokens = _eff_tok.filter_stream(_eff_tok.remap_byte_array(val_tokens))
    V = _eff_tok.vocab_size
    print(f"\nRemapped to token IDs (V={V})")
    print(f"  Train: {len(train_tokens):,} tokens after filtering")
    print(f"  Val:   {len(val_tokens):,} tokens after filtering")
else:
    V = 256  # vocab size for raw bytes

print(f"\nTrain tokens: {len(train_tokens):,}")
print(f"Val tokens:   {len(val_tokens):,}")
print(f"Vocab size V={V}")

### Investigations over all shards

In [ ]:
# Accumulate unigram and bigram counts across ALL shards (one at a time to save memory)
# Also collect letter-sequence counts (letter_seq = contiguous run of ASCII letters a-z/A-Z)
from collections import Counter

all_shard_files = sorted(glob.glob(train_pattern))
print(f"Streaming counts over {len(all_shard_files)} shards...")

full_unigram_counts = np.zeros(V, dtype=np.int64)
full_bigram_counts = np.zeros((V, V), dtype=np.int64)
full_total_tokens = 0
prev_last_byte = None  # to count the bigram spanning shard boundaries

# Letter-sequence counting
letter_seq_counts = Counter()
is_letter = np.zeros(256, dtype=bool)
is_letter[65:91] = True  # A-Z
is_letter[97:123] = True  # a-z
pending_seq = b""  # partial word carried across shard boundary

for i, f in enumerate(all_shard_files):
    shard = load_data_shard(Path(f))
    n = len(shard)
    full_total_tokens += n

    # Unigram
    np.add.at(full_unigram_counts, shard, 1)

    # Bigram: cross-boundary pair from previous shard
    if prev_last_byte is not None:
        full_bigram_counts[prev_last_byte, shard[0]] += 1

    # Bigram: within shard
    pairs = shard[:-1].astype(np.int64) * V + shard[1:].astype(np.int64)
    np.add.at(full_bigram_counts.ravel(), pairs, 1)

    # Words: find runs of letters, handle shard boundary
    letter_mask = is_letter[shard]
    # Find boundaries where letter/non-letter transitions occur
    transitions = np.diff(letter_mask.astype(np.int8))
    starts = np.where(transitions == 1)[0] + 1  # non-letter -> letter
    ends = np.where(transitions == -1)[0] + 1  # letter -> non-letter

    # Handle shard starting with a letter (continuation of pending_seq or new word)
    if letter_mask[0]:
        starts = np.concatenate([[0], starts])
    # Handle shard ending with a letter (word continues into next shard)
    if letter_mask[-1]:
        ends = np.concatenate([ends, [n]])

    for s, e in zip(starts, ends):
        seq_bytes = bytes(shard[s:e])
        if s == 0 and pending_seq:
            # Continuation from previous shard
            seq_bytes = pending_seq + seq_bytes
            pending_seq = b""
        if e == n and letter_mask[-1]:
            # Word continues into next shard
            pending_seq = seq_bytes
        else:
            letter_seq_counts[seq_bytes.lower()] += 1

    # If shard starts with non-letter, flush any pending word from prev shard
    if not letter_mask[0] and pending_seq:
        letter_seq_counts[pending_seq.lower()] += 1
        pending_seq = b""

    prev_last_byte = int(shard[-1])
    sz = Path(f).stat().st_size / 1e6
    print(
        f"  [{i + 1:2d}/{len(all_shard_files)}] {Path(f).name} — {n:>12,} tokens ({sz:.0f} MB)  "
        f"cumulative: {full_total_tokens:,}  letter_seqs so far: {sum(letter_seq_counts.values()):,} "
        f"unique letter_seqs: {len(letter_seq_counts):,}"
    )
    del shard  # free memory

# Flush final pending word
if pending_seq:
    letter_seq_counts[pending_seq.lower()] += 1

total_letter_seqs = sum(letter_seq_counts.values())
print(f"\nTotal tokens across all shards: {full_total_tokens:,}")
print(f"Unique bytes seen: {(full_unigram_counts > 0).sum()} / {V}")
print(f"Unique bigrams seen: {(full_bigram_counts > 0).sum():,} / {V * V:,}")
print(f"Total letter sequences: {total_letter_seqs:,}")
print(f"Unique letter sequences (case-insensitive): {len(letter_seq_counts):,}")

In [ ]:
# plot distributin of leter sequence lengths
seq_lengths = [len(seq) for seq in letter_seq_counts.keys()]
max_len = max(seq_lengths)
print(f"Max letter sequence length: {max_len}")
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(seq_lengths, bins=np.arange(1, max_len) - 0.5, density=True)
ax.set_title("Distribution of Letter Sequence Lengths")
ax.set_xlabel("Length of Letter Sequence")
ax.set_ylabel("Proportion of Sequences")
ax.set_yscale("log")

In [ ]:
print("\nLongest letter sequences:")
sorted_seqs = sorted(letter_seq_counts.keys(), key=len, reverse=True)
for seq in sorted_seqs[:100]:  # top 100 longest
    print(f"  (length {len(seq)}, count {letter_seq_counts[seq]}) {seq.decode()} ")


In [ ]:
print("\nTop 50 letter sequences:")
for rank, (seq, count) in enumerate(letter_seq_counts.most_common(500000), 1):
    if count <= 5:
        break  # stop at singletons
    pct = count / total_letter_seqs * 100
    print(f"  {rank:3d}. {seq.decode():>15s}: {count:>12,}  ({pct:5.7f}%)")

In [ ]:
print("\Bottom 50 letter sequences:")
for rank, (seq, count) in enumerate(letter_seq_counts.most_common()[-500:], 1):
    pct = count / total_letter_seqs * 100
    print(f"  {rank:3d}. {seq.decode():>15s}: {count:>12,}  ({pct:5.2f}%)")

In [ ]:
# Decompose long letter sequences into known words using greedy longest-match
#
# 1. Build core vocab: sequences of length 2..MAX_WORD_LEN with freq >= MIN_FREQ
# 2. For sequences longer than MAX_WORD_LEN, greedily match core words from the left
#    (longest first), then try from the right on any unmatched remainder
# 3. Collect all pieces into a cleaned counter

MAX_WORD_LEN = 25  # core words are at most this long
MIN_FREQ = 50  # minimum count to be considered a core word
MIN_MATCH_LEN = 2  # don't greedily match single letters (too aggressive)
DECOMPOSE_ABOVE = 25  # only try to decompose sequences longer than this

# Step 1: Build core vocabulary (sorted by length descending for greedy matching)
core_vocab = set()
for seq, count in letter_seq_counts.items():
    slen = len(seq)
    if MIN_MATCH_LEN <= slen <= MAX_WORD_LEN and count >= MIN_FREQ:
        core_vocab.add(seq)

# Sort by length descending so greedy matching prefers longer words
core_vocab_sorted = sorted(core_vocab, key=len, reverse=True)
print(
    f"Core vocabulary: {len(core_vocab_sorted):,} entries "
    f"(len {MIN_MATCH_LEN}–{MAX_WORD_LEN}, freq >= {MIN_FREQ})"
)

# Build a set for O(1) lookup (we'll iterate by length for greedy)
# Group by length for efficient matching
max_len = max(len(w) for w in core_vocab_sorted) if core_vocab_sorted else 0
vocab_by_len = {}
for w in core_vocab_sorted:
    vocab_by_len.setdefault(len(w), set()).add(w)
available_lengths = sorted(vocab_by_len.keys(), reverse=True)  # longest first


def greedy_segment_left(seq_bytes):
    """Segment from left using greedy longest-match. Returns list of (piece, matched) tuples."""
    pieces = []
    pos = 0
    while pos < len(seq_bytes):
        matched = False
        for wlen in available_lengths:
            if wlen > len(seq_bytes) - pos:
                continue
            candidate = seq_bytes[pos : pos + wlen]
            if candidate in vocab_by_len.get(wlen, set()):
                pieces.append((candidate, True))
                pos += wlen
                matched = True
                break
        if not matched:
            # Collect unmatched bytes one at a time, merge consecutive later
            pieces.append((seq_bytes[pos : pos + 1], False))
            pos += 1
    return pieces


def greedy_segment_right(seq_bytes):
    """Segment from right using greedy longest-match. Returns list of (piece, matched) tuples."""
    pieces = []
    pos = len(seq_bytes)
    while pos > 0:
        matched = False
        for wlen in available_lengths:
            if wlen > pos:
                continue
            candidate = seq_bytes[pos - wlen : pos]
            if candidate in vocab_by_len.get(wlen, set()):
                pieces.append((candidate, True))
                pos -= wlen
                matched = True
                break
        if not matched:
            pieces.append((seq_bytes[pos - 1 : pos], False))
            pos -= 1
    pieces.reverse()
    return pieces


def merge_unmatched(pieces):
    """Merge consecutive unmatched single bytes into one piece."""
    merged = []
    buf = b""
    for piece, matched in pieces:
        if not matched:
            buf += piece
        else:
            if buf:
                merged.append((buf, False))
                buf = b""
            merged.append((piece, True))
    if buf:
        merged.append((buf, False))
    return merged


def segment(seq_bytes):
    """Try left-greedy first. For any unmatched remainders, try right-greedy."""
    left_pieces = merge_unmatched(greedy_segment_left(seq_bytes))

    # For unmatched pieces, try right-greedy
    final = []
    for piece, matched in left_pieces:
        if matched or len(piece) <= MIN_MATCH_LEN:
            final.append((piece, matched))
        else:
            right_pieces = merge_unmatched(greedy_segment_right(piece))
            final.extend(right_pieces)
    return final


# Step 2: Decompose and recount
cleaned_seq_counts = Counter()
n_decomposed = 0
n_short = 0
decompose_examples = []

for seq, count in letter_seq_counts.items():
    if len(seq) <= DECOMPOSE_ABOVE:
        # Keep as-is
        cleaned_seq_counts[seq] += count
        n_short += 1
    else:
        pieces = segment(seq)
        if len(pieces) == 1 and not pieces[0][1]:
            # No matches found at all — keep original
            cleaned_seq_counts[seq] += count
        else:
            n_decomposed += 1
            if n_decomposed <= 20:
                decompose_examples.append((seq, count, pieces))
            for piece, _matched in pieces:
                cleaned_seq_counts[piece] += count

total_cleaned = sum(cleaned_seq_counts.values())
print(f"\nSequences kept as-is (len <= {DECOMPOSE_ABOVE}): {n_short:,}")
print(f"Sequences decomposed: {n_decomposed:,}")
print(f"Unique letter sequences before: {len(letter_seq_counts):,}")
print(f"Unique letter sequences after:  {len(cleaned_seq_counts):,}")
print(f"Total count before: {sum(letter_seq_counts.values()):,}")
print(f"Total count after:  {total_cleaned:,}")

print(f"\nExample decompositions:")
for seq, count, pieces in decompose_examples[:20]:
    parts_str = " | ".join((p.decode() + ("✓" if m else "?")) for p, m in pieces)
    print(f"  {seq.decode()[:60]:60s} (n={count:,}) → {parts_str}")

print(f"\nTop 50 cleaned letter sequences:")
for rank, (seq, count) in enumerate(cleaned_seq_counts.most_common(50), 1):
    pct = count / total_cleaned * 100
    print(f"  {rank:3d}. {seq.decode():>15s}: {count:>12,}  ({pct:5.2f}%)")

In [ ]:
unused_tokens = []
for token_id, count in enumerate(full_unigram_counts):
    if count == 0:
        unused_tokens.append(token_id)
print(f"\nUnused byte values (train): {len(unused_tokens)} / {V}")
print(f"Unused byte values: {unused_tokens}")

## Peek at the data as text

In [ ]:
# Decode first 500 bytes as UTF-8
def _decode_tokens(tokens):
    if _eff_tok is not None:
        return _eff_tok.decode_to_str(tokens)
    return bytes(tokens).decode("utf-8", errors="replace")


print("=== Train (first 500 bytes) ===")
print(_decode_tokens(train_tokens[:500]))
print("\n=== Val (first 500 bytes) ===")
print(_decode_tokens(val_tokens[:500]))

## Unigram statistics

In [ ]:
# Build counts from train
unigram_counts = np.bincount(train_tokens, minlength=V)
n_unique_uni = (unigram_counts > 0).sum()
total_train = unigram_counts.sum()

print(f"Unique byte values seen (train): {n_unique_uni} / {V} ({n_unique_uni / V:.1%})")
print(f"Total train bytes: {total_train:,}")
print()


def _token_label(idx):
    """Human-readable label for a token ID."""
    if _eff_tok is not None:
        if idx < _eff_tok.n_special:
            return f"<sp{idx}>"
        raw = int(_eff_tok._id_to_byte[idx])
        ch = chr(raw) if 32 <= raw < 127 else f"0x{raw:02x}"
        return f"{ch}(b{raw})"
    ch = chr(idx) if 32 <= idx < 127 else f"0x{idx:02x}"
    return ch


# Top 30 unigrams
ranked = np.argsort(unigram_counts)[::-1]
print("Top 30 bytes (train):")
for i, idx in enumerate(ranked[:30]):
    count = unigram_counts[idx]
    pct = count / total_train * 100
    label = _token_label(idx)
    print(f"  {i + 1:2d}. token {idx:3d} ({label:>10s}): {count:>12,}  ({pct:5.2f}%)")

# Train entropy (for reference)
probs_train = unigram_counts / total_train
probs_nz = probs_train[probs_train > 0]
unigram_entropy_train = -np.sum(probs_nz * np.log2(probs_nz))

# Validation BPB: cross-entropy of val data under train unigram distribution
# H_cross = -1/N * sum_i log2(P_train(val_i))
log2_probs = np.full(V, -np.inf)
log2_probs[probs_train > 0] = np.log2(probs_train[probs_train > 0])
val_log2_probs = log2_probs[val_tokens]
# Handle unseen bytes in val
n_unseen = (val_log2_probs == -np.inf).sum()
if n_unseen > 0:
    print(f"\nWARNING: {n_unseen} val bytes unseen in train — using Laplace smoothing")
    probs_smoothed = (unigram_counts + 1) / (total_train + V)
    val_log2_probs = np.log2(probs_smoothed[val_tokens])

unigram_entropy_bits = -val_log2_probs.mean()

print(f"\nUnigram train entropy: {unigram_entropy_train:.4f} BPB")
print(f"Unigram val cross-entropy (BPB): {unigram_entropy_bits:.4f}")

In [ ]:
unused_tokens = []
for token_id, count in enumerate(unigram_counts):
    if count == 0:
        unused_tokens.append(token_id)
print(f"\nUnused byte values (train): {len(unused_tokens)} / {V}")
print(f"Unused byte values: {unused_tokens}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Distribution (log scale)
ax = axes[0]
ax.bar(range(V), unigram_counts, width=1.0, color="steelblue", alpha=0.7)
ax.set_yscale("log")
ax.set_xlabel("Byte value")
ax.set_ylabel("Count (log)")
ax.set_title("Unigram byte distribution (train)")
ax.set_xlim(0, V)

# CDF
ax = axes[1]
sorted_counts = np.sort(unigram_counts[unigram_counts > 0])[::-1]
cdf = np.cumsum(sorted_counts) / total_train
ax.plot(range(len(cdf)), cdf, color="darkorange", linewidth=2)
ax.set_xlabel("Rank")
ax.set_ylabel("Cumulative probability")
ax.set_title("Unigram CDF (by rank)")
ax.axhline(0.9, color="gray", linestyle="--", alpha=0.5, label="90%")
ax.axhline(0.99, color="gray", linestyle=":", alpha=0.5, label="99%")
ax.legend()

plt.tight_layout()
plt.show()

## Bigram statistics

In [ ]:
# Build bigram counts from train data
bigram_matrix = np.zeros((V, V), dtype=np.int64)
pairs = train_tokens[:-1].astype(np.int64) * V + train_tokens[1:].astype(np.int64)
np.add.at(bigram_matrix.ravel(), pairs, 1)

n_unique_bi = (bigram_matrix > 0).sum()
total_bigrams_train = bigram_matrix.sum()

print(
    f"Unique bigrams (train): {n_unique_bi:,} / {V * V:,} ({n_unique_bi / (V * V):.2%})"
)
print(f"Total bigram occurrences (train): {total_bigrams_train:,}")

# Top 30 bigrams
flat_idx = np.argsort(bigram_matrix.ravel())[::-1]
print("\nTop 30 bigrams (train):")
for i in range(30):
    idx = flat_idx[i]
    a, b = divmod(idx, V)
    count = bigram_matrix[a, b]
    pct = count / total_bigrams_train * 100
    ca = chr(a) if 32 <= a < 127 else f"0x{a:02x}"
    cb = chr(b) if 32 <= b < 127 else f"0x{b:02x}"
    print(f"  {i + 1:2d}. ({ca}, {cb}): {count:>12,}  ({pct:5.2f}%)")

# Validation BPB: cross-entropy of val bigrams under train conditional distribution
# P(next|prev) from train counts
row_sums = bigram_matrix.sum(axis=1, keepdims=True)
with np.errstate(divide="ignore", invalid="ignore"):
    cond_probs = np.where(row_sums > 0, bigram_matrix / row_sums, 0)

# Evaluate on val
val_prev = val_tokens[:-1].astype(np.int64)
val_next = val_tokens[1:].astype(np.int64)
val_cond_p = cond_probs[val_prev, val_next]

# Handle unseen bigrams with backoff to unigram + smoothing
n_unseen_bi = (val_cond_p == 0).sum()
if n_unseen_bi > 0:
    print(
        f"\nNote: {n_unseen_bi} val bigrams unseen in train ({n_unseen_bi / len(val_prev):.4%}) — backoff to smoothed unigram"
    )
    # For unseen bigrams, use smoothed unigram probability
    probs_smoothed_uni = (unigram_counts + 1) / (total_train + V)
    backoff_p = probs_smoothed_uni[val_next]
    val_cond_p = np.where(val_cond_p > 0, val_cond_p, backoff_p)

val_log2_p = np.log2(val_cond_p)
bigram_cond_entropy_bits = -val_log2_p.mean()

# Train conditional entropy for reference
with np.errstate(divide="ignore", invalid="ignore"):
    log_cond = np.where(cond_probs > 0, np.log2(cond_probs), 0)
p_prev = unigram_counts / total_train
cond_entropy_per_row = -np.sum(cond_probs * log_cond, axis=1)
bigram_train_entropy = np.sum(p_prev * cond_entropy_per_row)

print(f"\nBigram train conditional entropy: {bigram_train_entropy:.4f} BPB")
print(f"Bigram val cross-entropy (BPB): {bigram_cond_entropy_bits:.4f}")
print(
    f"Reduction from unigram: {unigram_entropy_bits - bigram_cond_entropy_bits:.4f} bits"
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

try:
    # use full_bigram_counts if available (from streaming over all shards), otherwise use bigram_matrix from train
    bigram_matrix_to_use = full_bigram_counts
except NameError:
    bigram_matrix_to_use = bigram_matrix

# Bigram heatmap (log scale)
ax = axes[0]
with np.errstate(divide="ignore"):
    log_matrix = np.log10(bigram_matrix_to_use.astype(float))
    log_matrix[bigram_matrix_to_use == 0] = np.nan
im = ax.imshow(log_matrix, aspect="auto", cmap="viridis", origin="lower")
ax.set_xlabel("Next byte")
ax.set_ylabel("Previous byte")
ax.set_title("Bigram counts (log10)")
plt.colorbar(im, ax=ax, label="log10(count)")

# Fanout distribution
ax = axes[1]
fanouts = (bigram_matrix_to_use > 0).sum(axis=1)
ax.bar(range(V), fanouts, width=1.0, color="seagreen", alpha=0.7)
ax.set_xlabel("Byte value (context)")
ax.set_ylabel("Number of distinct successors")
ax.set_title("Bigram fanout per byte")
ax.set_xlim(-1, V)

plt.tight_layout()
plt.show()

print(
    f"\nFanout stats: min={fanouts[fanouts > 0].min()}, median={np.median(fanouts[fanouts > 0]):.0f}, "
    f"mean={fanouts[fanouts > 0].mean():.1f}, max={fanouts.max()}"
)

## Trigram statistics
With V=256, the full trigram table is 256^3 = 16.7M entries — we use sparse counting.

In [ ]:
from collections import Counter, defaultdict

# Build trigram counts from train
N_train = len(train_tokens)
a = train_tokens[:-2].astype(np.int64)
b = train_tokens[1:-1].astype(np.int64)
c = train_tokens[2:].astype(np.int64)
trigram_keys = a * (V * V) + b * V + c

trigram_ids, trigram_counts_arr = np.unique(trigram_keys, return_counts=True)
n_unique_tri = len(trigram_ids)
total_trigrams_train = trigram_counts_arr.sum()

print(
    f"Unique trigrams (train): {n_unique_tri:,} / {V**3:,} ({n_unique_tri / V**3:.4%})"
)
print(f"Total trigram occurrences (train): {total_trigrams_train:,}")

# Top 30 trigrams
top_idx = np.argsort(trigram_counts_arr)[::-1][:30]
print("\nTop 30 trigrams (train):")
for i, idx in enumerate(top_idx):
    key = trigram_ids[idx]
    count = trigram_counts_arr[idx]
    a_val = key // (V * V)
    b_val = (key // V) % V
    c_val = key % V
    tri_bytes = bytes([a_val, b_val, c_val])
    tri_repr = tri_bytes.decode("utf-8", errors="replace")
    pct = count / total_trigrams_train * 100
    print(
        f"  {i + 1:2d}. ({a_val:3d},{b_val:3d},{c_val:3d}) {tri_repr!r:>8s}: {count:>10,}  ({pct:5.3f}%)"
    )

# Build lookup: trigram_key -> count, and bigram context totals
trigram_count_dict = dict(zip(trigram_ids.tolist(), trigram_counts_arr.tolist()))

bigram_context_totals = defaultdict(int)
for tri_id, tri_count in zip(trigram_ids, trigram_counts_arr):
    ctx = int(tri_id // V)
    bigram_context_totals[ctx] += int(tri_count)

# Train conditional entropy (for reference)
trigram_train_entropy = 0.0
for tri_id, tri_count in zip(trigram_ids, trigram_counts_arr):
    ctx = int(tri_id // V)
    p_c_given_ab = tri_count / bigram_context_totals[ctx]
    trigram_train_entropy -= (tri_count / total_trigrams_train) * np.log2(p_c_given_ab)

# Validation BPB: evaluate val trigrams under train distribution
val_a = val_tokens[:-2].astype(np.int64)
val_b = val_tokens[1:-1].astype(np.int64)
val_c = val_tokens[2:].astype(np.int64)
val_trigram_keys = val_a * (V * V) + val_b * V + val_c
val_bigram_ctx = val_a * V + val_b

# Vectorized lookup: for each val trigram, get P(c|a,b) from train
# Build dense bigram matrix for fallback (already have it from bigram cell)
n_val_tri = len(val_trigram_keys)
val_log2_p = np.zeros(n_val_tri, dtype=np.float64)
n_unseen_tri = 0
n_backoff_bi = 0

for i in range(n_val_tri):
    tri_key = int(val_trigram_keys[i])
    ctx_key = int(val_bigram_ctx[i])
    tri_count = trigram_count_dict.get(tri_key, 0)
    ctx_total = bigram_context_totals.get(ctx_key, 0)

    if tri_count > 0 and ctx_total > 0:
        val_log2_p[i] = np.log2(tri_count / ctx_total)
    else:
        # Backoff to bigram P(c|b)
        b_val = int(val_b[i])
        c_val = int(val_c[i])
        bi_p = cond_probs[b_val, c_val]
        if bi_p > 0:
            val_log2_p[i] = np.log2(bi_p)
            n_backoff_bi += 1
        else:
            # Backoff to smoothed unigram
            probs_smoothed_uni = (unigram_counts + 1) / (total_train + V)
            val_log2_p[i] = np.log2(probs_smoothed_uni[c_val])
            n_unseen_tri += 1

trigram_cond_entropy_bits = -val_log2_p.mean()

print(f"\nTrigram train conditional entropy: {trigram_train_entropy:.4f} BPB")
print(f"Trigram val cross-entropy (BPB): {trigram_cond_entropy_bits:.4f}")
if n_backoff_bi > 0 or n_unseen_tri > 0:
    print(f"  Backoff to bigram: {n_backoff_bi:,} ({n_backoff_bi / n_val_tri:.4%})")
    print(f"  Backoff to unigram: {n_unseen_tri:,} ({n_unseen_tri / n_val_tri:.4%})")
print(
    f"Reduction from bigram: {bigram_cond_entropy_bits - trigram_cond_entropy_bits:.4f} bits"
)

## 4-gram, 5-gram, and 6-gram statistics

In [ ]:
def _encode_ngram_keys(tokens, order, V=V):
    """Encode n-gram keys from a token array."""
    n = len(tokens)
    if order <= 7:
        key = np.zeros(n - order + 1, dtype=np.int64)
        for i in range(order):
            key += tokens[i : n - order + 1 + i].astype(np.int64) * (
                V ** (order - 1 - i)
            )
        return key
    elif order == 8:
        key = np.zeros(n - order + 1, dtype=np.uint64)
        for i in range(order):
            key = key * np.uint64(V) + tokens[i : n - order + 1 + i].astype(np.uint64)
        return key
    else:
        ngram_mat = np.empty((n - order + 1, order), dtype=np.uint8)
        for i in range(order):
            ngram_mat[:, i] = tokens[i : n - order + 1 + i]
        dt = np.dtype([("bytes", np.uint8, (order,))])
        return ngram_mat.view(dt).ravel()


def _decode_ngram_key(key, order, V=V):
    """Decode a single n-gram key back to byte values."""
    if order <= 8:
        k = int(key)
        vals = []
        for _ in range(order):
            vals.append(k % V)
            k //= V
        vals.reverse()
        return vals
    else:
        return list(key["bytes"])


def compute_ngram_stats(
    train_tokens, val_tokens, order, V=V, top_k=30, prev_order_val_fn=None
):
    """Compute n-gram stats on train, evaluate cross-entropy on val with backoff.

    prev_order_val_fn: callable(context_key, next_byte) -> log2(P) for backoff.
        If None, backs off to smoothed unigram.
    """
    # --- Train: build n-gram counts ---
    train_keys = _encode_ngram_keys(train_tokens, order, V)
    ngram_ids, ngram_counts = np.unique(train_keys, return_counts=True)

    n_unique = len(ngram_ids)
    total_train = ngram_counts.sum()
    possible = V**order

    print(
        f"Unique {order}-grams (train): {n_unique:,} / {possible:,} ({n_unique / possible:.6%})"
    )
    print(f"Total {order}-gram occurrences (train): {total_train:,}")

    # Top-k from train
    top_idx = np.argsort(ngram_counts)[::-1][:top_k]
    print(f"\nTop {top_k} {order}-grams (train):")
    for i, idx in enumerate(top_idx):
        vals = _decode_ngram_key(ngram_ids[idx], order, V)
        ngram_bytes = bytes(vals)
        ngram_repr = ngram_bytes.decode("utf-8", errors="replace")
        pct = ngram_counts[idx] / total_train * 100
        print(
            f"  {i + 1:2d}. {ngram_repr!r:>{order + 8}s}: {ngram_counts[idx]:>10,}  ({pct:5.3f}%)"
        )

    # --- Train conditional entropy ---
    if order <= 7:
        context_keys = ngram_ids // V
    elif order == 8:
        context_keys = ngram_ids // np.uint64(V)
    else:
        ctx_dt = np.dtype([("bytes", np.uint8, (order - 1,))])
        context_keys = ngram_ids["bytes"][:, :-1].copy().view(ctx_dt).ravel()

    ctx_ids, inverse = np.unique(context_keys, return_inverse=True)
    context_totals = np.zeros(len(ctx_ids), dtype=np.int64)
    np.add.at(context_totals, inverse, ngram_counts)
    p_given_ctx = ngram_counts / context_totals[inverse]
    train_cond_entropy = -np.sum((ngram_counts / total_train) * np.log2(p_given_ctx))

    # --- Build lookup for val evaluation ---
    # For orders <= 8, use dict for fast lookup
    if order <= 8:
        ngram_count_dict = dict(zip(ngram_ids.tolist(), ngram_counts.tolist()))
        ctx_total_dict = dict(zip(ctx_ids.tolist(), context_totals.tolist()))
    else:
        # For structured arrays, convert to tuple keys
        ngram_count_dict = {
            tuple(ngram_ids[i]["bytes"]): int(ngram_counts[i])
            for i in range(len(ngram_ids))
        }
        ctx_total_dict = {
            tuple(ctx_ids[i]["bytes"]): int(context_totals[i])
            for i in range(len(ctx_ids))
        }

    # --- Val cross-entropy ---
    val_keys = _encode_ngram_keys(val_tokens, order, V)
    n_val = len(val_keys)
    val_log2_p = np.zeros(n_val, dtype=np.float64)
    n_backoff = 0

    # Smoothed unigram as final fallback
    probs_smoothed_uni = (unigram_counts + 1) / (total_train + V)
    log2_uni = np.log2(probs_smoothed_uni)

    if order <= 8:
        # Also encode val context keys for lookup
        val_ctx_keys = _encode_ngram_keys(val_tokens, order - 1, V)
        # val_ctx_keys has length len(val_tokens) - order + 2
        # val_keys has length len(val_tokens) - order + 1
        # context for val_keys[i] is val_ctx_keys[i]
        val_next_bytes = val_tokens[order - 1 :][:n_val]

        for i in range(n_val):
            nk = int(val_keys[i]) if order <= 7 else int(val_keys[i])
            ck = int(val_ctx_keys[i]) if order - 1 <= 7 else int(val_ctx_keys[i])
            nc = ngram_count_dict.get(nk, 0)
            ct = ctx_total_dict.get(ck, 0)

            if nc > 0 and ct > 0:
                val_log2_p[i] = np.log2(nc / ct)
            elif prev_order_val_fn is not None:
                val_log2_p[i] = prev_order_val_fn(i)
                n_backoff += 1
            else:
                val_log2_p[i] = log2_uni[int(val_next_bytes[i])]
                n_backoff += 1
    else:
        val_next_bytes = val_tokens[order - 1 :][:n_val]
        for i in range(n_val):
            nk = tuple(val_keys[i]["bytes"])
            ck = nk[:-1]
            nc = ngram_count_dict.get(nk, 0)
            ct = ctx_total_dict.get(ck, 0)

            if nc > 0 and ct > 0:
                val_log2_p[i] = np.log2(nc / ct)
            elif prev_order_val_fn is not None:
                val_log2_p[i] = prev_order_val_fn(i)
                n_backoff += 1
            else:
                val_log2_p[i] = log2_uni[int(val_next_bytes[i])]
                n_backoff += 1

    val_cond_entropy = -val_log2_p.mean()

    print(f"\n{order}-gram train conditional entropy: {train_cond_entropy:.4f} BPB")
    print(f"{order}-gram val cross-entropy (BPB): {val_cond_entropy:.4f}")
    if n_backoff > 0:
        print(f"  Backoff: {n_backoff:,} / {n_val:,} ({n_backoff / n_val:.4%})")

    return (
        n_unique,
        total_train,
        ngram_ids,
        ngram_counts,
        val_cond_entropy,
        train_cond_entropy,
        ngram_count_dict,
        ctx_total_dict,
    )


ngram_results = {}
max_order = 8

for order in range(4, max_order + 1):
    print(f"{'=' * 60}")
    print(f"  {order}-gram analysis")
    print(f"{'=' * 60}")

    # Build backoff function from previous order's lookup
    if order == 4:
        # Backoff to trigram (use trigram_count_dict and bigram_context_totals from trigram cell)
        def make_backoff_trigram():
            val_tri_keys = _encode_ngram_keys(val_tokens, 3, V)
            val_bi_ctx = _encode_ngram_keys(val_tokens, 2, V)
            val_next = val_tokens[2:][: len(val_tri_keys)]
            probs_sm = (unigram_counts + 1) / (total_train + V)
            log2_sm = np.log2(probs_sm)

            def backoff_fn(i):
                # i is index into order-4 val, need to map to order-3 index
                # val n-gram of order k starts at position i, context is positions i..i+k-2
                # For order 4, position i maps to trigram starting at i+1
                j = i + 1
                if j < len(val_tri_keys):
                    tk = int(val_tri_keys[j])
                    ck = int(val_bi_ctx[j])
                    nc = trigram_count_dict.get(tk, 0)
                    ct = bigram_context_totals.get(ck, 0)
                    if nc > 0 and ct > 0:
                        return np.log2(nc / ct)
                bv = int(val_next[j]) if j < len(val_next) else 0
                return log2_sm[bv]

            return backoff_fn

        prev_backoff = make_backoff_trigram()
    elif order > 4:
        # Use previous order's dicts for backoff
        prev_nc_dict = prev_ngram_count_dict
        prev_ct_dict = prev_ctx_total_dict
        prev_o = order - 1

        def make_backoff(pnc, pct, po):
            val_prev_keys = _encode_ngram_keys(val_tokens, po, V)
            val_prev_ctx = _encode_ngram_keys(val_tokens, po - 1, V)
            val_next = val_tokens[po - 1 :][: len(val_prev_keys)]
            probs_sm = (unigram_counts + 1) / (total_train + V)
            log2_sm = np.log2(probs_sm)

            def backoff_fn(i):
                j = i + 1
                if j < len(val_prev_keys):
                    if po <= 8:
                        nk = int(val_prev_keys[j])
                        ck = int(val_prev_ctx[j])
                    else:
                        nk = tuple(val_prev_keys[j]["bytes"])
                        ck = nk[:-1]
                    nc = pnc.get(nk, 0)
                    ct = pct.get(ck, 0)
                    if nc > 0 and ct > 0:
                        return np.log2(nc / ct)
                bv = int(val_next[j]) if j < len(val_next) else 0
                return log2_sm[bv]

            return backoff_fn

        prev_backoff = make_backoff(prev_nc_dict, prev_ct_dict, prev_o)

    result = compute_ngram_stats(
        train_tokens, val_tokens, order, prev_order_val_fn=prev_backoff
    )
    (
        n_unique,
        total,
        ids,
        counts,
        val_h,
        train_h,
        prev_ngram_count_dict,
        prev_ctx_total_dict,
    ) = result

    ngram_results[order] = {
        "n_unique": n_unique,
        "total": total,
        "cond_entropy": val_h,
        "train_entropy": train_h,
    }
    if order == 4:
        n_unique_quad = n_unique
        quadgram_cond_entropy_bits = val_h
    print()

## Plot pairwise average distance between tokens

In [ ]:
# For each pair (a, b): among all positions where token[i]=a,
# how far is the next occurrence of b?  i.e. min{j>i : token[j]=b} - i
# We compute mean, median (p50), and min distance → three 256x256 heatmaps.

N_DIST = min(2_000_000, len(train_tokens))  # subset for speed
tokens_sub = train_tokens[:N_DIST]
print(f"Computing pairwise distances on {N_DIST:,} bytes...")

# Step 1: for each byte value b, compute dist_to_next[b, i] = distance from i to next b
# Stored as (V, N_DIST) uint32 array. ~512MB for 2M tokens — fine.
# Actually let's be memory-efficient: compute per-b and aggregate immediately.

mean_dist = np.full((V, V), np.nan, dtype=np.float32)
min_dist = np.full((V, V), np.nan, dtype=np.float32)
p50_dist = np.full((V, V), np.nan, dtype=np.float32)
p10_dist = np.full((V, V), np.nan, dtype=np.float32)

# Precompute positions for each source byte a
positions_of = {}
for a in range(V):
    pos = np.where(tokens_sub == a)[0]
    if len(pos) > 0:
        positions_of[a] = pos

# For each target byte b, compute distance-to-next-b array, then gather stats
for b in range(V):
    # dist_to_next_b[i] = how far from position i to the next occurrence of b
    b_positions = np.where(tokens_sub == b)[0]
    if len(b_positions) == 0:
        continue

    # For each position i, next occurrence of b = b_positions[searchsorted(b_positions, i+1)]
    # We process per source byte a for efficiency
    for a, a_positions in positions_of.items():
        # For each a_pos, find the next b_pos > a_pos (skip a_pos itself for a==b)
        search_start = a_positions + (1 if a == b else 0)
        idx = np.searchsorted(b_positions, search_start, side="left")
        # Filter out positions where there's no next b
        valid = idx < len(b_positions)
        if not valid.any():
            continue
        distances = b_positions[idx[valid]] - a_positions[valid]
        mean_dist[a, b] = distances.mean()
        min_dist[a, b] = distances.min()
        p50_dist[a, b] = np.median(distances)
        p10_dist[a, b] = np.percentile(distances, 10)

print("Done.")
print(f"Pairs with data: {(~np.isnan(mean_dist)).sum():,} / {V * V:,}")

# Plot
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

for ax, data, title in [
    (axes[0, 0], mean_dist, "Mean distance to next occurrence"),
    (axes[0, 1], p50_dist, "Median (p50) distance"),
    (axes[1, 0], p10_dist, "p10 distance"),
    (axes[1, 1], min_dist, "Min distance"),
]:
    with np.errstate(divide="ignore"):
        log_data = np.log10(data.astype(float))
        log_data[np.isnan(data)] = np.nan
    im = ax.imshow(log_data, aspect="auto", cmap="viridis", origin="lower")
    ax.set_xlabel("Target byte (b)")
    ax.set_ylabel("Source byte (a)")
    ax.set_title(title)
    plt.colorbar(im, ax=ax, label="log10(distance)")

plt.suptitle(
    f"Pairwise byte distances (first {N_DIST / 1e6:.0f}M bytes)", fontsize=14, y=1.01
)
plt.tight_layout()
plt.show()

# Also show some interesting stats
print("\nClosest pairs (min distance = 1, i.e. always adjacent):")
always_adj = np.argwhere(min_dist == 1)
if len(always_adj) > 0:
    # Sort by mean distance (most likely to be adjacent)
    sorted_by_mean = always_adj[np.argsort([mean_dist[a, b] for a, b in always_adj])]
    for a, b in sorted_by_mean[:20]:
        ca = chr(a) if 32 <= a < 127 else f"0x{a:02x}"
        cb = chr(b) if 32 <= b < 127 else f"0x{b:02x}"
        print(
            f"  ({ca}, {cb}): mean={mean_dist[a, b]:.1f}, median={p50_dist[a, b]:.1f}, min={min_dist[a, b]:.0f}"
        )

print("\nMost distant pairs (highest mean distance):")
valid_mask = ~np.isnan(mean_dist)
if valid_mask.any():
    flat_idx = np.argsort(mean_dist[valid_mask])[::-1]
    coords = np.argwhere(valid_mask)
    for i in range(min(10, len(flat_idx))):
        a, b = coords[flat_idx[i]]
        ca = chr(a) if 32 <= a < 127 else f"0x{a:02x}"
        cb = chr(b) if 32 <= b < 127 else f"0x{b:02x}"
        print(
            f"  ({ca}, {cb}): mean={mean_dist[a, b]:.1f}, median={p50_dist[a, b]:.1f}, min={min_dist[a, b]:.0f}"
        )

## Token correlation via KS test against geometric null
For each pair (a, b), compute the KS statistic between the observed distance
distribution (a → next b) and the Geometric(p_b) null that assumes independent placement.

In [ ]:
# KS statistic: max|F_empirical(d) - F_geometric(d)| where F_geo(d) = 1 - (1-p_b)^d
# Uses the same positions_of[] and token subset from the heatmap cell above.

MIN_SAMPLES = 30  # skip pairs with too few distances

# Precompute p_b from the token subset used for distances
token_subset = train_tokens[:N_DIST]
freq = np.bincount(token_subset, minlength=V).astype(np.float64)
p = freq / len(token_subset)  # per-byte frequency

ks_matrix = np.full((V, V), np.nan)
mean_ratio_matrix = np.full((V, V), np.nan)  # observed_mean / (1/p_b)
n_samples_matrix = np.zeros((V, V), dtype=np.int64)
empty = np.array([], dtype=np.int64)

for a in range(V):
    a_pos = positions_of.get(a, empty)
    if len(a_pos) == 0:
        continue
    for b in range(V):
        b_pos = positions_of.get(b, empty)
        if len(b_pos) == 0 or p[b] == 0:
            continue

        # Distance from each a to next b (searchsorted)
        idx = np.searchsorted(b_pos, a_pos, side="right")
        valid = idx < len(b_pos)
        if valid.sum() < MIN_SAMPLES:
            continue

        distances = b_pos[idx[valid]] - a_pos[valid]
        distances = distances[
            distances > 0
        ]  # exclude distance 0 (same position if a==b)
        n = len(distances)
        if n < MIN_SAMPLES:
            continue

        n_samples_matrix[a, b] = n

        # Mean ratio: observed / expected
        expected_mean = 1.0 / p[b]
        mean_ratio_matrix[a, b] = distances.mean() / expected_mean

        # KS statistic against Geometric(p_b)
        # For sorted distances d, F_geo(d) = 1 - (1-p_b)^d
        d_sorted = np.sort(distances)
        ecdf = np.arange(1, n + 1) / n
        # Use log to avoid underflow for (1-p)^d when p is small
        log_survival = d_sorted * np.log1p(-p[b])  # log((1-p)^d)
        geo_cdf = 1.0 - np.exp(log_survival)

        # Two-sided KS: max of |F_emp - F_geo| at each step and just before each step
        ks = max(
            np.max(np.abs(ecdf - geo_cdf)),
            np.max(np.abs((ecdf - 1.0 / n) - geo_cdf)),
        )
        ks_matrix[a, b] = ks

    if a % 32 == 0:
        print(f"  row {a}/256...")

print("Done.")

# --- Summary stats ---
valid = ~np.isnan(ks_matrix)
print(f"\nValid pairs: {valid.sum():,} / {V * V:,}")
print(
    f"KS statistic: median={np.nanmedian(ks_matrix):.4f}, "
    f"mean={np.nanmean(ks_matrix):.4f}, "
    f"max={np.nanmax(ks_matrix):.4f}"
)
print(
    f"Mean ratio:   median={np.nanmedian(mean_ratio_matrix):.4f}, "
    f"mean={np.nanmean(mean_ratio_matrix):.4f}"
)

# --- Heatmaps ---
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

im0 = axes[0].imshow(
    ks_matrix,
    aspect="equal",
    cmap="inferno",
    vmin=0,
    vmax=np.nanpercentile(ks_matrix, 99),
)
axes[0].set_title("KS statistic vs Geometric(p_b) null")
axes[0].set_xlabel("target byte b")
axes[0].set_ylabel("source byte a")
plt.colorbar(im0, ax=axes[0], shrink=0.8)

im1 = axes[1].imshow(
    mean_ratio_matrix, aspect="equal", cmap="RdBu_r", vmin=0.5, vmax=1.5
)
axes[1].set_title("Mean distance ratio: observed / expected")
axes[1].set_xlabel("target byte b")
axes[1].set_ylabel("source byte a")
plt.colorbar(im1, ax=axes[1], shrink=0.8, label="ratio (< 1 = attract, > 1 = repel)")

plt.tight_layout()
plt.show()

# --- Top correlated pairs ---
print("\nTop 30 most correlated pairs (highest KS statistic):")
print(
    f"  {'a':>5s}  {'b':>5s}  {'a_chr':>6s}  {'b_chr':>6s}  {'KS':>8s}  {'ratio':>8s}  {'n_obs':>8s}"
)
print(f"  {'─' * 5}  {'─' * 5}  {'─' * 6}  {'─' * 6}  {'─' * 8}  {'─' * 8}  {'─' * 8}")

flat = ks_matrix.copy()
flat[np.isnan(flat)] = -1
top_idx = np.argsort(flat.ravel())[::-1][:30]
for idx in top_idx:
    a, b = divmod(idx, V)
    ca = chr(a) if 32 <= a < 127 else f"0x{a:02x}"
    cb = chr(b) if 32 <= b < 127 else f"0x{b:02x}"
    print(
        f"  {a:>5d}  {b:>5d}  {ca:>6s}  {cb:>6s}  {ks_matrix[a, b]:>8.4f}  "
        f"{mean_ratio_matrix[a, b]:>8.3f}  {n_samples_matrix[a, b]:>8,}"
    )

print("\nTop 30 attracting pairs (lowest mean ratio):")
print(
    f"  {'a':>5s}  {'b':>5s}  {'a_chr':>6s}  {'b_chr':>6s}  {'ratio':>8s}  {'KS':>8s}  {'n_obs':>8s}"
)
print(f"  {'─' * 5}  {'─' * 5}  {'─' * 6}  {'─' * 6}  {'─' * 8}  {'─' * 8}  {'─' * 8}")

flat_r = mean_ratio_matrix.copy()
flat_r[np.isnan(flat_r)] = 999
top_attract = np.argsort(flat_r.ravel())[:30]
for idx in top_attract:
    a, b = divmod(idx, V)
    ca = chr(a) if 32 <= a < 127 else f"0x{a:02x}"
    cb = chr(b) if 32 <= b < 127 else f"0x{b:02x}"
    print(
        f"  {a:>5d}  {b:>5d}  {ca:>6s}  {cb:>6s}  {mean_ratio_matrix[a, b]:>8.3f}  "
        f"{ks_matrix[a, b]:>8.4f}  {n_samples_matrix[a, b]:>8,}"
    )

In [ ]:
# Plot distance distributions for a sample of byte pairs
# Compare shapes: are they all roughly geometric/exponential, or do some differ?

# Pick interesting pairs: top bigrams, some rare ones, same-byte pairs
sample_pairs = []

# Top 10 most frequent bigrams
flat_idx_top = np.argsort(bigram_matrix.ravel())[::-1]
for i in range(10):
    a, b = divmod(flat_idx_top[i], V)
    sample_pairs.append((a, b, "frequent"))

# A few same-byte pairs (common ASCII)
for ch in [ord("e"), ord(" "), ord("t"), ord("\n")]:
    if ch in positions_of:
        sample_pairs.append((ch, ch, "self"))

# A few rare but existing pairs (high mean distance), but min occurrences > 100
valid_mask = ~np.isnan(mean_dist)
valid_mask &= bigram_matrix > 100  # ensure they exist in train
if valid_mask.any():
    coords = np.argwhere(valid_mask)
    sorted_by_mean_desc = coords[np.argsort([mean_dist[a, b] for a, b in coords])[::-1]]
    for a, b in sorted_by_mean_desc[:10]:
        sample_pairs.append((int(a), int(b), "rare"))

# Compute and plot actual distance distributions
fig, axes = plt.subplots(3, 1, figsize=(14, 12))

b_positions_cache = {}


def get_b_positions(b):
    if b not in b_positions_cache:
        b_positions_cache[b] = np.where(tokens_sub == b)[0]
    return b_positions_cache[b]


for ax, category, title in [
    (axes[0], "frequent", "Frequent bigrams"),
    (axes[1], "self", "Same-byte pairs (a→a)"),
    (axes[2], "rare", "Rare/distant pairs"),
]:
    pairs_in_cat = [(a, b) for a, b, cat in sample_pairs if cat == category]
    max_dist_plot = 500 if category != "rare" else 5000

    for a, b in pairs_in_cat:
        a_pos = positions_of.get(a)
        b_pos = get_b_positions(b)
        if a_pos is None or len(b_pos) == 0:
            continue
        search_start = a_pos + (1 if a == b else 0)
        idx = np.searchsorted(b_pos, search_start, side="left")
        valid = idx < len(b_pos)
        if not valid.any():
            continue
        distances = b_pos[idx[valid]] - a_pos[valid]

        # Histogram with log y-axis
        ca = chr(a) if 32 <= a < 127 else f"0x{a:02x}"
        cb = chr(b) if 32 <= b < 127 else f"0x{b:02x}"
        label = f"({ca},{cb}) μ={distances.mean():.0f}"
        if category == "rare":
            bins = np.logspace(
                0, np.log10(min(distances.max() + 1, max_dist_plot)), 100
            )
        else:
            bins = np.arange(
                0, min(distances.max() + 1, max_dist_plot), max(1, max_dist_plot // 200)
            )
        ax.hist(
            distances,
            bins=bins,
            alpha=0.5,
            density=True,
            label=label,
            histtype="step",
            linewidth=1.5,
        )

    if category == "rare":
        ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel("Distance (positions)")
    ax.set_ylabel("Density (log)")
    ax.set_title(title)
    ax.legend(fontsize=8, ncol=2)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Overlay all categories on one normalized plot to compare shapes
fig, ax = plt.subplots(figsize=(14, 5))
for a, b, cat in sample_pairs[:4] + sample_pairs[-4:]:  # first 8 to avoid clutter
    a_pos = positions_of.get(a)
    b_pos = get_b_positions(b)
    if a_pos is None or len(b_pos) == 0:
        continue
    search_start = a_pos + (1 if a == b else 0)
    idx = np.searchsorted(b_pos, search_start, side="left")
    valid = idx < len(b_pos)
    if not valid.any():
        continue
    distances = b_pos[idx[valid]] - a_pos[valid]
    # Normalize distances by mean to compare shapes
    normed = distances / distances.mean()
    ca = chr(a) if 32 <= a < 127 else f"0x{a:02x}"
    cb = chr(b) if 32 <= b < 127 else f"0x{b:02x}"
    bins = np.linspace(0, 5, 20)
    ax.hist(
        normed,
        bins=bins,
        alpha=0.4,
        density=True,
        # label=f"({ca},{cb}) [{cat}]",
        histtype="step",
        linewidth=1.5,
    )

# Reference: Exp(1) distribution (memoryless baseline)
x = np.linspace(0, 5, 100)
ax.plot(x, np.exp(-x), "k--", linewidth=2, label="Exp(1) reference")

ax.set_xlabel("Distance / mean distance")
ax.set_ylabel("Density")
ax.set_title("Normalized distance distributions (distance/mean) — do they collapse?")
ax.legend(fontsize=8, ncol=2)
ax.set_yscale("log")
ax.set_ylim(1e-3, 5)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Mixture model: correlated + uncorrelated components

Fit each byte-pair distance distribution as a mixture:

$$P(d) = w \cdot f_{\text{correlated}}(d) + (1-w) \cdot \text{Exp}(d; \lambda)$$

where the **uncorrelated** component is always exponential (memoryless baseline),
and we compare different choices for the **correlated** component:

| Model | Correlated component | Parameters | Can have mode > 0? |
|-------|---------------------|------------|-------------------|
| `exp+exp` | Exponential(λ₁) | 3 | No |
| `gamma+exp` | Gamma(k, θ) | 4 | Yes (k > 1) |
| `lognorm+exp` | LogNormal(μ, σ) | 4 | Yes |

We compare via **AIC** (lower = better, penalizes extra parameters).

In [ ]:
from scipy.stats import gamma as gamma_dist, lognorm as lognorm_dist
from scipy.optimize import minimize


def softmax(logits):
    """Stable softmax for mixing weights."""
    x = logits - logits.max()
    e = np.exp(x)
    return e / e.sum()


# --- Component definitions: each returns (log_pdf_array, n_params, description) ---


def _exp_logpdf(d, params):
    lam = params[0]
    return np.log(lam) - lam * d


def _exp_unpack(raw):
    return (np.exp(raw[0]),), 1


def _exp_init(mu):
    return [[np.log(1.0 / mu)], [np.log(2.0 / mu)], [np.log(0.5 / mu)]]


def _exp_bounds():
    return [(-12, 12)]


def _exp_describe(params):
    return f"Exp(μ={1 / params[0]:.1f})"


def _gamma_logpdf(d, params):
    k, theta = params
    return gamma_dist.logpdf(d, a=k, scale=theta)


def _gamma_unpack(raw):
    k = np.exp(raw[0]) + 0.1
    theta = np.exp(raw[1])
    return (k, theta), 2


def _gamma_init(mu):
    return [
        [np.log(2), np.log(mu / 8)],
        [np.log(4), np.log(mu / 16)],
        [np.log(1), np.log(mu / 4)],
        [np.log(8), np.log(mu / 32)],
    ]


def _gamma_bounds():
    return [(-2, 5), (-5, 15)]


def _gamma_describe(params):
    k, theta = params
    mode = max(0, (k - 1) * theta)
    return f"Γ(k={k:.2f},θ={theta:.1f},mode={mode:.1f},μ={k * theta:.1f})"


def _lognorm_logpdf(d, params):
    mu_ln, sigma_ln = params
    return lognorm_dist.logpdf(d, s=sigma_ln, scale=np.exp(mu_ln))


def _lognorm_unpack(raw):
    mu_ln = raw[0]
    sigma_ln = np.exp(raw[1]) + 0.01
    return (mu_ln, sigma_ln), 2


def _lognorm_init(mu):
    lm = np.log(mu)
    return [
        [lm - 1, np.log(0.5)],
        [lm, np.log(1.0)],
        [lm - 2, np.log(0.8)],
    ]


def _lognorm_bounds():
    return [(-5, 15), (-3, 4)]


def _lognorm_describe(params):
    mu_ln, sigma_ln = params
    mode = np.exp(mu_ln - sigma_ln**2)
    mean = np.exp(mu_ln + sigma_ln**2 / 2)
    return f"LN(mode={mode:.1f},μ={mean:.1f},σ={sigma_ln:.2f})"


COMPONENTS = {
    "exp": (_exp_logpdf, _exp_unpack, _exp_init, _exp_bounds, _exp_describe, 1),
    "gamma": (
        _gamma_logpdf,
        _gamma_unpack,
        _gamma_init,
        _gamma_bounds,
        _gamma_describe,
        2,
    ),
    "lognorm": (
        _lognorm_logpdf,
        _lognorm_unpack,
        _lognorm_init,
        _lognorm_bounds,
        _lognorm_describe,
        2,
    ),
}


class FlexMixture:
    """Flexible K-component mixture of exp/gamma/lognorm distributions."""

    def __init__(self, spec):
        """spec: list of component type names, e.g. ['lognorm', 'lognorm', 'exp']."""
        self.spec = spec
        self.K = len(spec)
        self.name = "+".join(sorted(spec))
        self.short_name = "+".join(
            f"{sum(1 for s in spec if s == t)}{t[0].upper()}"
            for t in dict.fromkeys(spec)
        )
        self.comp_defs = [COMPONENTS[s] for s in spec]
        # n_params: sum of component params + (K-1) mixing weight logits
        self.n_params = sum(c[5] for c in self.comp_defs) + (self.K - 1)

    def _unpack(self, raw):
        """Unpack flat parameter vector into (weights, list_of_component_params)."""
        # First K-1 values are weight logits
        if self.K > 1:
            weight_logits = np.array(raw[: self.K - 1].tolist() + [0.0])
            weights = softmax(weight_logits)
            idx = self.K - 1
        else:
            weights = np.array([1.0])
            idx = 0

        comp_params = []
        for _, unpack_fn, _, _, _, n_raw in self.comp_defs:
            params, consumed = unpack_fn(raw[idx : idx + n_raw])
            comp_params.append(params)
            idx += consumed
        return weights, comp_params

    def log_likelihood(self, d, raw):
        weights, comp_params = self._unpack(raw)
        # log P(d) = logsumexp over components
        log_terms = []
        for i, (logpdf_fn, *_) in enumerate(self.comp_defs):
            log_terms.append(
                np.log(np.clip(weights[i], 1e-10, 1)) + logpdf_fn(d, comp_params[i])
            )
        # Stack and logsumexp
        log_stack = np.array(log_terms)  # (K, N)
        return logsumexp_ax0(log_stack)

    def fit(self, distances, n_restarts=6):
        d = distances.astype(np.float64)
        d = np.maximum(d, 0.5)
        n = len(d)
        mu = d.mean()

        bounds = []
        # Weight logits
        for _ in range(self.K - 1):
            bounds.append((-6, 6))
        # Component params
        for _, _, _, bounds_fn, _, _ in self.comp_defs:
            bounds.extend(bounds_fn())

        # Generate initial points by combining component-level inits
        from itertools import product

        comp_inits = [init_fn(mu) for _, _, init_fn, _, _, _ in self.comp_defs]
        # Take a few combinations (limit to avoid explosion)
        all_combos = list(product(*comp_inits))
        np.random.seed(42)
        if len(all_combos) > n_restarts:
            selected = [
                all_combos[i]
                for i in np.random.choice(len(all_combos), n_restarts, replace=False)
            ]
        else:
            selected = all_combos

        best_result = None
        best_nll = np.inf

        for combo in selected:
            for w_init_scale in [0.0, 0.5, -0.5]:
                init = [w_init_scale] * (self.K - 1)
                for comp_raw in combo:
                    init.extend(comp_raw)
                init = np.array(init, dtype=np.float64)

                try:
                    res = minimize(
                        lambda p: -self.log_likelihood(d, p).mean(),
                        init,
                        method="L-BFGS-B",
                        bounds=bounds,
                        options={"maxiter": 500, "ftol": 1e-12},
                    )
                    if res.fun < best_nll:
                        best_nll = res.fun
                        best_result = res
                except Exception:
                    continue

        if best_result is None:
            return None

        weights, comp_params = self._unpack(best_result.x)
        ll = -best_result.fun
        aic = 2 * self.n_params - 2 * ll * n
        bic = self.n_params * np.log(n) - 2 * ll * n

        # Single exp baseline
        lam_s = 1.0 / mu
        ll_single = (np.log(lam_s) - lam_s * d).mean()

        return {
            "model": self.short_name,
            "spec": self.spec,
            "weights": weights,
            "comp_params": comp_params,
            "ll": ll,
            "aic": aic,
            "bic": bic,
            "ll_single": ll_single,
            "n_params": self.n_params,
        }

    def pdf(self, x, weights, comp_params):
        total = np.zeros_like(x, dtype=np.float64)
        for i, (logpdf_fn, *_) in enumerate(self.comp_defs):
            total += weights[i] * np.exp(logpdf_fn(x, comp_params[i]))
        return total

    def pdf_components(self, x, weights, comp_params):
        parts = []
        for i, (logpdf_fn, *_) in enumerate(self.comp_defs):
            parts.append(weights[i] * np.exp(logpdf_fn(x, comp_params[i])))
        return parts

    def describe(self, weights, comp_params):
        parts = []
        for i, (_, _, _, _, desc_fn, _) in enumerate(self.comp_defs):
            parts.append(f"  w={weights[i]:.3f} {desc_fn(comp_params[i])}")
        return "\n".join(parts)


def logsumexp_ax0(log_arr):
    """logsumexp along axis 0 of a 2D array."""
    mx = log_arr.max(axis=0)
    return mx + np.log(np.exp(log_arr - mx).sum(axis=0))


# Define model configurations to test
configs = [
    # ["exp"],
    # ["exp", "exp"],
    ["lognorm", "exp"],
    # ["gamma", "exp"],
    # ["lognorm", "lognorm", "exp"],
    # ["gamma", "gamma", "exp"],
    # ["lognorm", "gamma", "exp"],
    # ["exp", "exp", "exp"],
]

flex_models = [FlexMixture(spec) for spec in configs]

print(f"Testing {len(flex_models)} model configurations:")
for fm in flex_models:
    print(f"  {fm.short_name:>20s}  ({fm.n_params} params): {fm.spec}")

# Fit all pairs × all models
all_fits = []
for a, b, cat in sample_pairs:
    a_pos = positions_of.get(a)
    b_pos = get_b_positions(b)
    if a_pos is None or len(b_pos) == 0:
        continue
    search_start = a_pos + (1 if a == b else 0)
    idx = np.searchsorted(b_pos, search_start, side="left")
    valid = idx < len(b_pos)
    if not valid.any():
        continue
    distances = b_pos[idx[valid]] - a_pos[valid]
    if len(distances) < 100:
        continue

    ca = chr(a) if 32 <= a < 127 else f"0x{a:02x}"
    cb = chr(b) if 32 <= b < 127 else f"0x{b:02x}"
    pair_label = f"({ca},{cb})"
    print(f"  Fitting {pair_label}...", end="", flush=True)

    pf = {
        "pair": pair_label,
        "cat": cat,
        "a": a,
        "b": b,
        "distances": distances,
        "n": len(distances),
        "fits": {},
    }

    for fm in flex_models:
        result = fm.fit(distances)
        if result is not None:
            pf["fits"][fm.short_name] = result
    print(f" done ({len(pf['fits'])} models)")
    all_fits.append(pf)

# AIC comparison table
model_names = [fm.short_name for fm in flex_models]
print(f"\n{'Pair':>12s} {'Cat':>8s} │", end="")
for mn in model_names:
    print(f" {mn:>10s}", end="")
print(" │ Best")
print("─" * (25 + 11 * len(model_names) + 8))

for pf in all_fits:
    aics = {mn: pf["fits"][mn]["aic"] for mn in model_names if mn in pf["fits"]}
    if not aics:
        continue
    best = min(aics, key=aics.get)
    best_aic = aics[best]
    row = f"{pf['pair']:>12s} {pf['cat']:>8s} │"
    for mn in model_names:
        if mn in aics:
            delta = aics[mn] - best_aic
            marker = "★" if mn == best else " "
            row += f" {delta:>9.0f}{marker}"
        else:
            row += f" {'N/A':>10s}"
    row += f" │ {best}"
    print(row)

print(f"\nΔAIC relative to best (0★ = best). Lower = better, penalizes extra params.")

In [ ]:
# Plot best-fit decomposition for each pair
fm_lookup = {fm.short_name: fm for fm in flex_models}

# Color palette for components
comp_colors = {"exp": "steelblue", "gamma": "red", "lognorm": "purple"}

n_plots = min(len(all_fits), 12)
ncols = 3
nrows = (n_plots + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(5.5 * ncols, 4.5 * nrows))
axes_flat = np.array(axes).flatten()

for i, pf in enumerate(all_fits[:n_plots]):
    ax = axes_flat[i]
    d = pf["distances"]
    max_d = np.percentile(d, 99)
    bins = np.arange(0.5, max_d, max(1, max_d / 150))
    ax.hist(d, bins=bins, density=True, alpha=0.25, color="gray", label="Data")

    x = np.linspace(0.5, max_d, 500)
    aics = {mn: pf["fits"][mn]["aic"] for mn in pf["fits"]}
    if not aics:
        continue
    best_name = min(aics, key=aics.get)
    best_fit = pf["fits"][best_name]
    best_fm = fm_lookup[best_name]

    # Plot best model: total PDF
    pdf_total = best_fm.pdf(x, best_fit["weights"], best_fit["comp_params"])
    ax.plot(x, pdf_total, "k-", linewidth=2.5, label=f"{best_name} (best)", zorder=10)

    # Plot individual components
    parts = best_fm.pdf_components(x, best_fit["weights"], best_fit["comp_params"])
    for j, (part, comp_type) in enumerate(zip(parts, best_fm.spec)):
        w = best_fit["weights"][j]
        desc_fn = COMPONENTS[comp_type][4]
        desc = desc_fn(best_fit["comp_params"][j])
        color = comp_colors[comp_type]
        ax.plot(
            x,
            part,
            "--",
            color=color,
            linewidth=1.3,
            alpha=0.8,
            label=f"  w={w:.2f} {desc[:25]}",
        )

    # Also plot runner-up model (thin line) for comparison
    sorted_models = sorted(aics.items(), key=lambda kv: kv[1])
    if len(sorted_models) > 1:
        runner_name = sorted_models[1][0]
        runner_fit = pf["fits"][runner_name]
        runner_fm = fm_lookup[runner_name]
        pdf_runner = runner_fm.pdf(x, runner_fit["weights"], runner_fit["comp_params"])
        delta = sorted_models[1][1] - sorted_models[0][1]
        ax.plot(
            x,
            pdf_runner,
            color="orange",
            linewidth=1,
            alpha=0.5,
            label=f"{runner_name} (ΔAIC={delta:.0f})",
        )

    ax.set_yscale("log")
    ax.set_ylim(max(1e-8, pdf_total[pdf_total > 0].min() * 0.1), pdf_total.max() * 3)
    ax.set_title(f"{pf['pair']} [{pf['cat']}]", fontsize=10)
    ax.legend(fontsize=5.5, loc="upper right")
    ax.set_xlabel("Distance")
    ax.grid(True, alpha=0.2)

for j in range(n_plots, len(axes_flat)):
    axes_flat[j].set_visible(False)

plt.suptitle("Best mixture model decomposition per byte pair", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

# Detailed best-fit parameters
print("\n=== Best model details ===\n")
for pf in all_fits:
    aics = {mn: pf["fits"][mn]["aic"] for mn in pf["fits"]}
    if not aics:
        continue
    best_name = min(aics, key=aics.get)
    best_fit = pf["fits"][best_name]
    best_fm = fm_lookup[best_name]
    print(
        f"{pf['pair']:>12s} [{pf['cat']:>8s}]  best={best_name} ({best_fit['n_params']} params)"
    )
    print(best_fm.describe(best_fit["weights"], best_fit["comp_params"]))
    print(f"  LL={best_fit['ll']:.4f}  AIC={best_fit['aic']:.0f}")
    print()

# Count wins per model
print("=== Model win counts ===")
wins = {}
for pf in all_fits:
    aics = {mn: pf["fits"][mn]["aic"] for mn in pf["fits"]}
    if aics:
        best = min(aics, key=aics.get)
        wins[best] = wins.get(best, 0) + 1
for mn, count in sorted(wins.items(), key=lambda x: -x[1]):
    print(f"  {mn:>20s}: {count} wins")

## Coverage and storage summary

In [ ]:
all_entropies_val = [
    ("Uniform", np.log2(V)),
    ("Unigram", unigram_entropy_bits),
    ("Bigram", bigram_cond_entropy_bits),
    ("Trigram", trigram_cond_entropy_bits),
    *[
        (f"{order}-gram", ngram_results[order]["cond_entropy"])
        for order in range(4, max_order + 1)
    ],
]

all_entropies_train = [
    ("Uniform", np.log2(V)),
    ("Unigram", unigram_entropy_train),
    ("Bigram", bigram_train_entropy),
    ("Trigram", trigram_train_entropy),
    *[
        (f"{order}-gram", ngram_results[order]["train_entropy"])
        for order in range(4, max_order + 1)
    ],
]

all_uniques = [
    ("Unigram", 1, n_unique_uni),
    ("Bigram", 2, n_unique_bi),
    ("Trigram", 3, n_unique_tri),
    *[
        (f"{order}-gram", order, ngram_results[order]["n_unique"])
        for order in range(4, max_order + 1)
    ],
]

print(f"=== N-gram coverage (byte-level, V={V}) ===")
for name, order, n_unique in all_uniques:
    print(
        f"  {name:>8s}: {n_unique:>14,} / {V**order:>20,} = {n_unique / V**order:.10%}"
    )

print()
print("=== Dense storage (full tables, int32 counts) ===")
bpc = 4
for name, order, _ in all_uniques:
    size = V**order * bpc
    if size < 1e3:
        print(f"  {name:>8s}: {size:.0f} B")
    elif size < 1e6:
        print(f"  {name:>8s}: {size / 1e3:.1f} KB")
    elif size < 1e9:
        print(f"  {name:>8s}: {size / 1e6:.1f} MB")
    elif size < 1e12:
        print(f"  {name:>8s}: {size / 1e9:.1f} GB")
    else:
        print(f"  {name:>8s}: {size / 1e12:.1f} TB")

print()
print("=== Sparse storage (observed n-grams only, key + int32 count) ===")
for name, order, n_unique in all_uniques:
    entry_size = order + bpc
    size = n_unique * entry_size
    if size < 1e6:
        print(f"  {name:>8s}: {size / 1e3:.1f} KB  ({n_unique:,} entries)")
    elif size < 1e9:
        print(f"  {name:>8s}: {size / 1e6:.2f} MB  ({n_unique:,} entries)")
    else:
        print(f"  {name:>8s}: {size / 1e9:.2f} GB  ({n_unique:,} entries)")

print()
print(
    f"=== BPB summary (train={MAX_TOKENS_TRAIN / 1e6:.0f}M, val={MAX_TOKENS_VAL / 1e6:.0f}M bytes) ==="
)
print(f"  {'Model':>8s}  {'Train BPB':>10s}  {'Val BPB':>10s}  {'Overfit':>8s}")
print(f"  {'─' * 8}  {'─' * 10}  {'─' * 10}  {'─' * 8}")
for (name_t, h_t), (name_v, h_v) in zip(all_entropies_train, all_entropies_val):
    overfit = h_v - h_t
    print(f"  {name_t:>8s}  {h_t:>10.4f}  {h_v:>10.4f}  {overfit:>+8.4f}")

print()
print("=== Reduction per order (val BPB) ===")
for i in range(1, len(all_entropies_val)):
    prev_name, prev_h = all_entropies_val[i - 1]
    name, h = all_entropies_val[i]
    print(f"  {prev_name:>8s} -> {name:<8s}: {h - prev_h:+.4f} bits")

# Plot entropy vs n-gram order (train and val)
fig, ax = plt.subplots(figsize=(10, 5))
orders = list(range(len(all_entropies_val)))
labels = [name for name, _ in all_entropies_val]
val_values = [h for _, h in all_entropies_val]
train_values = [h for _, h in all_entropies_train]

ax.plot(
    orders,
    train_values,
    "s--",
    color="gray",
    linewidth=1.5,
    markersize=6,
    label="Train",
    alpha=0.7,
)
ax.plot(
    orders, val_values, "o-", color="steelblue", linewidth=2, markersize=8, label="Val"
)
for i, (lbl, val) in enumerate(zip(labels, val_values)):
    ax.annotate(
        f"{val:.3f}",
        (i, val),
        textcoords="offset points",
        xytext=(0, 10),
        ha="center",
        fontsize=9,
    )
ax.set_xticks(orders)
ax.set_xticklabels(labels, rotation=30, ha="right")
ax.set_ylabel("Conditional entropy (BPB)")
ax.set_title(
    f"Byte-level conditional entropy by n-gram order\n(train={MAX_TOKENS_TRAIN / 1e6:.0f}M, val={MAX_TOKENS_VAL / 1e6:.0f}M bytes, with backoff)"
)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("byte_ngram_entropy.png", dpi=300)
# plt.show()

## Case-agnostic experiment
Merge uppercase output probability into lowercase, then score each val byte
by looking up `P_merged(lower(byte))`. Both `A` and `a` get the same (higher)
merged probability — this measures BPB if the model didn't need to distinguish case.

In [ ]:
if _eff_tok is not None:
    print("Skipping case-agnostic experiment (requires raw byte IDs)")
else:
    # Case-agnostic scoring: merge P(Upper) into P(lower) in output distribution,
    # then score val byte c as P_merged(lower(c)).

    lower_map = np.arange(256, dtype=np.uint8)
    lower_map[65:91] = lower_map[65:91] + 32  # A-Z -> a-z

    # Target for scoring: lower(true_byte) — used to index into merged distribution
    val_target = lower_map[val_tokens]

    # --- Unigram ---
    probs_uni = unigram_counts / total_train
    probs_uni_merged = probs_uni.copy()
    for u in range(65, 91):
        probs_uni_merged[u + 32] += probs_uni_merged[u]
        probs_uni_merged[u] = 0
    # Smooth zeros
    probs_uni_merged = np.where(probs_uni_merged > 0, probs_uni_merged, 1e-12)
    probs_uni_merged /= probs_uni_merged.sum()

    uni_ca_bpb = -np.log2(probs_uni_merged[val_target]).mean()

    # --- Bigram ---
    # Merge output columns: P_merged(lower(c) | prev) = P(c|prev) + P(upper(c)|prev)
    bi_merged = cond_probs.copy()  # (256, 256) from bigram cell
    for u in range(65, 91):
        bi_merged[:, u + 32] += bi_merged[:, u]
        bi_merged[:, u] = 0

    va_prev = val_tokens[:-1].astype(np.int64)
    va_tgt = val_target[1:].astype(np.int64)
    va_p = bi_merged[va_prev, va_tgt]
    # Backoff unseen to merged unigram
    unseen = va_p == 0
    if unseen.any():
        va_p = np.where(unseen, probs_uni_merged[va_tgt], va_p)
    bi_ca_bpb = -np.log2(va_p).mean()

    # --- Trigram ---
    # For each (a,b,c_lower): count = count(a,b,c_lower) + count(a,b,C_upper)
    va_a = val_tokens[:-2].astype(np.int64)
    va_b = val_tokens[1:-1].astype(np.int64)
    va_c = val_target[2:].astype(np.int64)  # lowered target
    n_val = len(va_a)
    tri_lp = np.zeros(n_val, dtype=np.float64)
    log2_uni_m = np.log2(probs_uni_merged)

    for i in range(n_val):
        a, b, c = int(va_a[i]), int(va_b[i]), int(va_c[i])
        ctx = a * V + b
        ctx_total = bigram_context_totals.get(ctx, 0)
        if ctx_total > 0:
            count = trigram_count_dict.get(a * V * V + b * V + c, 0)
            if 97 <= c <= 122:
                count += trigram_count_dict.get(a * V * V + b * V + (c - 32), 0)
            if count > 0:
                tri_lp[i] = np.log2(count / ctx_total)
                continue
        # Backoff to merged bigram
        bp = bi_merged[b, c]
        if bp > 0:
            tri_lp[i] = np.log2(bp)
        else:
            tri_lp[i] = log2_uni_m[c]

    tri_ca_bpb = -tri_lp.mean()

    # --- Higher orders (4..max_order) ---
    # Rebuild n-gram dicts and merge last-byte counts
    higher_ca = {}
    for order in range(4, max_order + 1):
        print(f"  Computing {order}-gram case-agnostic BPB...")
        tr_keys = _encode_ngram_keys(train_tokens, order, V)
        ng_ids, ng_cnts = np.unique(tr_keys, return_counts=True)
        ng_dict = dict(zip(ng_ids.tolist(), ng_cnts.tolist()))

        if order <= 7:
            ctx_keys = ng_ids // V
        else:
            ctx_keys = ng_ids // np.uint64(V)
        ctx_uids, ctx_inv = np.unique(ctx_keys, return_inverse=True)
        ctx_tots = np.zeros(len(ctx_uids), dtype=np.int64)
        np.add.at(ctx_tots, ctx_inv, ng_cnts)
        ctx_dict = dict(zip(ctx_uids.tolist(), ctx_tots.tolist()))

        va_keys = _encode_ngram_keys(val_tokens, order, V)
        va_ctx = _encode_ngram_keys(val_tokens, order - 1, V)
        va_tgt_o = val_target[order - 1 :][: len(va_keys)]
        n_va = len(va_keys)
        lp = np.zeros(n_va, dtype=np.float64)

        for i in range(n_va):
            ck = int(va_ctx[i])
            ct = ctx_dict.get(ck, 0)
            c = int(va_tgt_o[i])

            if ct > 0:
                # Key with lowered last byte
                nk_base = int(va_keys[i])
                true_last = int(val_tokens[order - 1 + i])
                nk_low = nk_base - true_last + c  # replace last byte with lower(c)
                count = ng_dict.get(nk_low, 0)
                if 97 <= c <= 122:
                    nk_up = nk_base - true_last + (c - 32)
                    count += ng_dict.get(nk_up, 0)
                if count > 0:
                    lp[i] = np.log2(count / ct)
                    continue

            # Backoff to merged bigram
            b_byte = int(val_tokens[order - 2 + i])
            bp = bi_merged[b_byte, c]
            if bp > 0:
                lp[i] = np.log2(bp)
            else:
                lp[i] = log2_uni_m[c]

        higher_ca[order] = -lp.mean()
        print(f"    {order}-gram: {higher_ca[order]:.4f} BPB")

    # --- Summary table ---
    print(f"\n{'=' * 70}")
    print(f"  Case-agnostic output merging — Val BPB comparison")
    print(f"{'=' * 70}")
    print(
        f"  {'Model':>8s}  {'Original':>10s}  {'Case-agn.':>10s}  {'Δ BPB':>10s}  {'% saved':>10s}"
    )
    print(f"  {'─' * 8}  {'─' * 10}  {'─' * 10}  {'─' * 10}  {'─' * 10}")

    rows = [
        ("Unigram", unigram_entropy_bits, uni_ca_bpb),
        ("Bigram", bigram_cond_entropy_bits, bi_ca_bpb),
        ("Trigram", trigram_cond_entropy_bits, tri_ca_bpb),
    ] + [
        (f"{o}-gram", ngram_results[o]["cond_entropy"], higher_ca[o])
        for o in range(4, max_order + 1)
    ]

    for name, orig, ca in rows:
        delta = ca - orig
        pct = -delta / orig * 100  # positive = saved
        print(
            f"  {name:>8s}  {orig:>10.4f}  {ca:>10.4f}  {delta:>+10.4f}  {pct:>+9.2f}%"
        )

    # How much of val is uppercase?
    n_upper = np.sum((val_tokens >= 65) & (val_tokens <= 90))
    print(
        f"\nUppercase bytes in val: {n_upper:,} / {len(val_tokens):,} = {n_upper / len(val_tokens):.2%}"
    )
    print(
        f"Theoretical max saving (uniform case): {np.log2(2):.4f} BPB × {n_upper / len(val_tokens):.4f} = {np.log2(2) * n_upper / len(val_tokens):.4f} BPB"
    )

## Lowercase experiment: how much does case cost?

## Comparison note: byte-level vs sp1024

| Property | Byte-level | sp1024 |
|----------|-----------|--------|
| Vocab size | 256 | 1024 |
| Dense bigram table | 256 KB | 4 MB |
| Dense trigram table | 64 MB | 4 GB |
| Dense 4-gram table | 16 GB | 4 TB |
| Metric | Bits per byte (BPB) | Nats per token |

Byte-level n-gram tables are dramatically smaller, making dense storage feasible up to trigrams.
However, byte-level models need to capture longer-range dependencies to match subword-level performance,
since each subword token corresponds to ~2-3 bytes on average.